In [ ]:
from typing import Dict, List, Union
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import shutil

In [ ]:
# Constants
BASE_PATH = os.path.join("D:\\", "Workspaces", "vscode-workspace", "ai_x_medicine", "data")
GDP_DATASET_FILENAME = "GDP.csv"
GINI_DATASET_FILENAME = "gini-coefficient-by-country-2025.csv"

RES_GDP_PATH = os.path.join(BASE_PATH, "out", "res-gdp.csv")
RES_GINI_PATH = os.path.join(BASE_PATH, "out", "res-gini.csv")
OUTPUT_PATH = os.path.join(BASE_PATH, "out", "socioeconomic-pain.csv")

In [ ]:
def remap_country_name():
    # todo
    pass

In [ ]:
# Util functions
def is_valid_data(val):
    if pd.notnull(val) and str(val).strip() != '':
        return True
    return False

def compute_pain(gdp_val: float, gini_val: float) -> float:
    if gini_val is None:
        return gdp_val
    else:
        return gini_val

def bar_plot(dataframe: pd.DataFrame):
    plt.figure(figsize=(12, 6))
    dataframe.plot(kind='bar')
    plt.xlabel('Countries')
    plt.ylabel('values')
    # plt.title('')
    plt.tight_layout()
    plt.show()

In [ ]:
# Download data
if not os.path.exists(os.path.join(BASE_PATH, GDP_DATASET_FILENAME)):
    path = kagglehub.dataset_download("annafabris/world-gdp-by-country-1960-2022")
    shutil.move(path, BASE_PATH)

In [ ]:
actual_gdp_path = os.path.join(BASE_PATH, "2", GDP_DATASET_FILENAME)    # the 2 is probably for some weird api download reason
gdp_df = pd.read_csv(actual_gdp_path)

In [ ]:
FIRST_GDP_COLUMN = 3
results: List[Dict[str, Union[str, int, int]]] = []
for idx, row in gdp_df.iterrows():
    country = row["Country"]
    data_point = None
    for col in gdp_df.columns[FIRST_GDP_COLUMN:]:
        val = row[col]
        # Check for non-empty and non-NaN
        if pd.notnull(val) and str(val).strip() != '':
            data_point = {
                'Country': country,
                'First Year': int(col),
                'First Value': int(val)
            }
            break
    if data_point is None:
        print(f"{country} has no GDP data")
        results.append({ 'Country': country, 'First Year': -1, 'First Value': -1 })
    else:
        print(f"{country} with first data at {data_point['First Year']}")
        results.append(data_point)


In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Count occurrences of each 'First Year'
year_counts = results_df['First Year'].value_counts().sort_index()
print(year_counts)

# Plot
plt.figure(figsize=(12, 6))
year_counts.plot(kind='bar')
plt.xlabel('First Year with GDP Data')
plt.ylabel('Number of Countries')
plt.title('Number of Countries by First Year with GDP Data')
plt.tight_layout()
plt.show()

In [ ]:
FIRST_GDP_COLUMN = 3
my_year_counts: Dict[int, int] = {}
for idx, row in gdp_df.iterrows():
    country = row["Country"]
    data_point = None
    for col in gdp_df.columns[FIRST_GDP_COLUMN:]:
        val = row[col]
        # Check for non-empty and non-NaN
        if pd.notnull(val) and str(val).strip() != '':
            if col not in my_year_counts:
                my_year_counts[col] = 0
            my_year_counts[col] += 1


In [ ]:
print(my_year_counts)

# ############################################################################################

In [ ]:
# [x] GDP: extract latest values by country and save it as csv
FIRST_GDP_COLUMN = 3
actual_gdp_path = os.path.join(BASE_PATH, "2", GDP_DATASET_FILENAME)    # the 2 is probably for some weird api download reason
gdp_df = pd.read_csv(actual_gdp_path)
gdp_results: List[Dict[str, Union[str, int, int]]] = []
for _, row in gdp_df.iterrows():
    country = row["Country"]
    data_point = None
    for idx in range(len(gdp_df.columns)-1, FIRST_GDP_COLUMN-1, -1):
        col = gdp_df.columns[idx]
        val = row[col]
        # Check for non-empty and non-NaN
        if pd.notnull(val) and str(val).strip() != '':
            data_point = {
                'Country': country,
                'Last Year': int(col),
                'Last Value': int(val)
            }
            break
    if data_point is None:
        print(f"{country} has no GDP data")
    else:
        print(f"{country} with last data at {data_point['Last Year']}")
        gdp_results.append(data_point)
res_gdp_df = pd.DataFrame(gdp_results)
res_gdp_df.to_csv(RES_GDP_PATH, index=False)


In [ ]:
# transform GDP data to [0, 1]
# description: relative logarithm of how often a countries GDP fits into the richest country's GDP
def transform_gdp(gdp_df: pd.DataFrame, verbose: bool = False) -> pd.DataFrame:
    gdp_df = gdp_df.copy(deep=True)
    max_gdp, min_gdp = gdp_df['Last Value'].max(), gdp_df['Last Value'].min()
    if verbose:
        print(f"Max value at {gdp_df[gdp_df['Last Value'] >= max_gdp]}")
    
    max_log_value = np.log(max_gdp / min_gdp)
    gdp_df['Last Value'] = gdp_df['Last Value'].apply(lambda x: np.log(max_gdp / x) / max_log_value)
    if verbose: 
        print(gdp_df)
    return gdp_df


In [ ]:
# transform GINI data to [0, 1]
# description: just divide by 100 as it is already in percent
def transform_gini(gini_df: pd.DataFrame, verbose: bool = False) -> pd.DataFrame:
    #gini_df = pd.read_csv(os.path.join(BASE_PATH, GINI_DATASET_FILENAME))
    gini_transformed = []
    for _, row in gini_df.iterrows():
        country = row["country"]
        dp1 = row["GiniCoefficient_GiniCoefficientViaWorldBank_gini_2024update"]
        dp2 = row["GiniCoefficient_GiniCoefficientViaCIA_gini_2024update"]
        if is_valid_data(dp1) and is_valid_data(dp2):
            gini_transformed.append({
                "Country": country,
                "gini": np.mean([dp1, dp2]) / 100
            })
        elif is_valid_data(dp1):
            gini_transformed.append({
                "Country": country,
                "gini": dp1 / 100
            })
        elif is_valid_data(dp2):
            gini_transformed.append({
                "Country": country,
                "gini": dp2 / 100
            })
    return pd.DataFrame(gini_transformed)

In [ ]:
# pain data
gdp_df = transform_gdp(pd.read_csv(RES_GDP_PATH, index_col=False))
gini_df = pd.read_csv(RES_GINI_PATH, index_col=False)

# gdp, gini index, pain killers?, 
socioeconomic_results: List[Dict[str, Union[str, float]]] = []
for country in gdp_df.Country:
    # consider gdp value
    row = gdp_df.loc[gdp_df.Country == country]
    gdp_val = row['Last Value'].values[0]
    
    # consider gini value
    row = gini_df.loc[gini_df.Country == country]
    if row.empty:
        gini_val = None
    else:
        gini_val = row['gini'].values[0]

    # calculate and store pain value
    pain_val = compute_pain(gdp_val, gini_val)
    socioeconomic_results.append({ "Country": country, "pain": pain_val })

socioeconomic_df = pd.DataFrame(socioeconomic_results)
socioeconomic_df.to_csv(OUTPUT_PATH, index=False)

In [ ]:
# Compare gini and gdp pain
df_gdp = transform_gdp(pd.read_csv(RES_GDP_PATH, index_col=False))
df_gini = pd.read_csv(RES_GINI_PATH, index_col=False)

comp_results: List[Dict[str, Union[str, float]]] = []
for country in df_gdp.Country:
    # consider gdp value
    row = df_gdp.loc[df_gdp.Country == country]
    gdp_val = row['Last Value'].values[0]
    
    # consider gini value
    row = df_gini.loc[df_gini.Country == country]
    if row.empty:
        gini_val = None
    else:
        gini_val = row['gini'].values[0]
        comp_results.append({ "Country": country, "gdp": gdp_val, "gini": gini_val })

df_comp = pd.DataFrame(comp_results)
bar_plot(df_comp)

In [ ]:
# Find intersection of country/location names
common_data = set(df_gdp["Country"]).intersection(set(df_gini["Country"]))
print(common_data)

# Subset both DataFrames to only those in the intersection
df_gini_sub = df_gini[df_gini["Country"].isin(common_data)].copy()
df_gdp_sub = df_gdp[df_gdp["Country"].isin(common_data)].copy()

In [ ]:
# Correlation between GINI and GDP (adapted)
plt.scatter(df_gini_sub.gini, df_gdp_sub['Last Value'])
plt.show()

In [ ]:
DISEASE_DATASET_PATH = os.path.join(BASE_PATH, "painful_disease_prevalence_vs_environment.csv")
df_diseases = pd.read_csv(DISEASE_DATASET_PATH)

df_gdp = pd.read_csv(RES_GDP_PATH, index_col=False)
df_gini = pd.read_csv(RES_GINI_PATH, index_col=False)

#print(df_diseases)

# Find intersection of country/location names
common_gdp = set(df_diseases["location"]).intersection(set(df_gdp["Country"]))
common_gini = set(df_diseases["location"]).intersection(set(df_gini["Country"]))

# Subset both DataFrames to only those in the intersection
df_diseases_gdp_sub = df_diseases[df_diseases["location"].isin(common_gdp)].copy()
df_gdp_sub = df_gdp[df_gdp["Country"].isin(common_gdp)].copy()
df_diseases_gini_sub = df_diseases[df_diseases["location"].isin(common_gini)].copy()
df_gini_sub = df_gini[df_gini["Country"].isin(common_gini)].copy()

print(f"common_gdp.size = {len(common_gdp)}")
print(f"df_gdp_sub.size = {df_gdp_sub.size}")
print(f"df_diseases_gdp_sub.size = {df_diseases_gdp_sub.size}")
print()
print(f"common_gini.size = {len(common_gini)}")
print(f"df_gini_sub.size = {df_gini_sub.size}")
print(f"df_diseases_gini_sub.size = {df_diseases_gini_sub.size}")


In [ ]:
# 
for column in df_diseases_gini_sub.columns:
    xval = df_diseases_gini_sub[column]
    yval = df_gini_sub["gini"]
    plt.scatter(xval, yval)
    plt.title(f"GINI vs {column}")
    plt.show()

In [ ]:
# plot GDP vs. various individuals' pains
for column in df_diseases_gdp_sub.columns:
    xval = df_diseases_gdp_sub[column]
    yval = df_gdp_sub["Last Value"]
    plt.scatter(xval, yval)
    plt.title(f"GDP vs {column}")
    plt.show()

In [ ]:
# Plot socioeconomic pain
plt.figure(figsize=(12, 6))
socioeconomic_df.plot(kind='bar')
plt.xlabel('Countries')
plt.ylabel('Socioeconomic Pain')
# plt.title('')
plt.tight_layout()
plt.show()

# PLAYGROUND

In [ ]:
transform_gdp(pd.read_csv(RES_GDP_PATH, index_col=False), verbose=True)

## Transform GDP to range [0, 1]

In [ ]:
# transform gdp data to [0, 1]
gdp_df = pd.read_csv(RES_GDP_PATH)
max_gdp = gdp_df['Last Value'].max()
min_gdp = gdp_df['Last Value'].min()
max_log_value = np.log(max_gdp / min_gdp)

new_gdp = gdp_df['Last Value'].apply(lambda x: x / max_gdp)
new_gdp_log = gdp_df['Last Value'].apply(lambda x: np.log(max_gdp / x) / max_log_value)
print(min_gdp)
print(max_log_value)

In [ ]:
f_new_gdp = pd.DataFrame(new_gdp)
plt.figure(figsize=(20, 6))
df_new_gdp.plot(kind='bar')
plt.xlabel('Country')
plt.ylabel('GDP ratio')
plt.title('No Log()')
plt.show()

In [ ]:
df_new_gdp_log = pd.DataFrame(new_gdp_log)
plt.figure(figsize=(20, 6))
df_new_gdp_log.plot(kind='bar')
plt.xlabel('Country')
plt.ylabel('GDP ratio')
plt.title('No Log()')
plt.show()

## extract conflict data

In [ ]:
CONFLICT_DATASET_PATH = os.path.join(BASE_PATH, "conflict-deaths-by-country.csv")
df_conflict = pd.read_csv(CONFLICT_DATASET_PATH)

In [ ]:
res_conflict = { }
for _, row in df_conflict.iterrows:
    country = row["location"]
    year = row["year"]
    deaths = row["val"]
    if country in res_conflict:
        # continue